# 00 — Environment Check

```text
Reviewer concern addressed: None directly; establishes the reproducibility baseline
    (pinned versions, input hashes, HiMaLAYAS source identification) that every later
    revision notebook depends on. Supports frozen-plan Phase 0 and rebuts the
    "resubmission = new submission" scrutiny on reproducibility.
Input files: data/yeast/gi_pcc_sampled.tsv, data/yeast/gi_score_sampled.tsv,
    data/yeast/go_bp_name_to_orfs.json, data/yeast/yeast_essential_orfs.txt,
    data/interdisciplinary/WorldWideDishes_2024_June.xlsx
    (all five inputs consumed across the three submitted root notebooks)
HiMaLAYAS version: determined in Section 3 below (this is the point of the notebook)
Random seed: none required here; this notebook establishes DEFAULT_RANDOM_SEED
    policy for later stochastic notebooks (10/20/21)
Primary parameters: none (no clustering or enrichment is run in this notebook)
Outputs written: revision/outputs/manifests/00_environment_check_manifest.json
Interpretation: see the Readiness Summary in the final section
```

Goal: establish the execution environment so every later notebook can import the
same manifest conventions (frozen plan, Phase 0).

## 1. Locate repo root and import shared revision utilities

Repo root is resolved by walking upward from the current working directory, since Jupyter's kernel CWD may be either the notebook's own folder (`revision/notebooks/`) or the repo root, depending on how it is launched.

In [1]:
import sys
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    markers = ("himalayas_src", "data", ".git", "revision")
    for candidate in (start, *start.parents):
        if all((candidate / marker).exists() for marker in markers):
            return candidate
    raise RuntimeError(f"Could not locate himalayas-publication repo root above {start}")


REPO_ROOT = _find_repo_root(Path.cwd().resolve())
SRC_DIR = REPO_ROOT / "revision" / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Kernel CWD:    {Path.cwd()}")
print(f"Repo root:     {REPO_ROOT}")
print(f"revision/src:  {SRC_DIR} (added to sys.path)")

Kernel CWD:    /Users/irahorecka/Desktop/harddrive_desktop/PhD/University of Toronto/Rost Lab/GitHub/himalayas-publication/revision/notebooks
Repo root:     /Users/irahorecka/Desktop/harddrive_desktop/PhD/University of Toronto/Rost Lab/GitHub/himalayas-publication
revision/src:  /Users/irahorecka/Desktop/harddrive_desktop/PhD/University of Toronto/Rost Lab/GitHub/himalayas-publication/revision/src (added to sys.path)


In [2]:
import json
import re
from datetime import datetime, timezone

import pandas as pd

from revision_utils import (
    DEFAULT_RANDOM_SEED,
    find_repo_root,
    git_info,
    himalayas_diagnostics,
    package_versions,
    python_info,
    revision_layout,
    sha256_file,
    write_manifest,
)

# Sanity check: revision_utils' own repo-root resolution agrees with the bootstrap cell above.
assert (
    find_repo_root() == REPO_ROOT
), "revision_utils resolved a different repo root than the bootstrap cell"

layout = revision_layout(REPO_ROOT)
for key, path in layout.items():
    print(f"{key:24s} {path}")

repo_root                /Users/irahorecka/Desktop/harddrive_desktop/PhD/University of Toronto/Rost Lab/GitHub/himalayas-publication
data_dir                 /Users/irahorecka/Desktop/harddrive_desktop/PhD/University of Toronto/Rost Lab/GitHub/himalayas-publication/data
revision_dir             /Users/irahorecka/Desktop/harddrive_desktop/PhD/University of Toronto/Rost Lab/GitHub/himalayas-publication/revision
outputs_dir              /Users/irahorecka/Desktop/harddrive_desktop/PhD/University of Toronto/Rost Lab/GitHub/himalayas-publication/revision/outputs
manifests_dir            /Users/irahorecka/Desktop/harddrive_desktop/PhD/University of Toronto/Rost Lab/GitHub/himalayas-publication/revision/outputs/manifests
tables_dir               /Users/irahorecka/Desktop/harddrive_desktop/PhD/University of Toronto/Rost Lab/GitHub/himalayas-publication/revision/outputs/tables
figures_dir              /Users/irahorecka/Desktop/harddrive_desktop/PhD/University of Toronto/Rost Lab/GitHub/himalayas

## 2. Repository identity (`himalayas-publication`)

Branch, commit, and working-tree cleanliness of this repo, for provenance.

In [3]:
pub_git = git_info(layout["repo_root"])
print(json.dumps(pub_git, indent=2))

{
  "is_git_repo": true,
  "branch": "first-revision",
  "commit": "eaeb0b63c589845d6dee9b30d216ca2c7b5c6c22",
  "describe": "v0.0.15-3-geaeb0b6",
  "is_dirty": true,
  "dirty_files": [
    "?? .DS_Store",
    "?? PREVIEW_NOTES.md",
    "?? himalayas_src/",
    "?? png/",
    "?? preview-release-notes.sh",
    "?? revision/"
  ]
}


## 3. Python interpreter and platform

In [4]:
python_info_result = python_info()
print(json.dumps(python_info_result, indent=2))

{
  "version": "3.12.8 (main, Jun 25 2025, 10:19:26) [Clang 17.0.0 (clang-1700.0.13.5)]",
  "version_info": [
    3,
    12,
    8,
    "final",
    0
  ],
  "executable": "/Users/irahorecka/.pyenv/versions/3.12.8/envs/himalayas/bin/python",
  "platform": "macOS-15.7.3-arm64-arm-64bit",
  "machine": "arm64"
}


## 4. Key dependency versions

Runtime dependencies actually imported by HiMaLAYAS and the submitted notebooks (`numpy`, `pandas`, `scipy`, `matplotlib`, `openpyxl`), plus the notebook-execution stack used to run/record these revision notebooks.

In [5]:
DEPENDENCY_NAMES = [
    "numpy",
    "pandas",
    "scipy",
    "matplotlib",
    "openpyxl",
    "ipykernel",
    "jupyterlab",
    "nbformat",
    "nbclient",
]
dep_versions = package_versions(DEPENDENCY_NAMES)
pd.DataFrame(sorted(dep_versions.items()), columns=["package", "version"])

,package,version
0,ipykernel,7.1.0
1,jupyterlab,4.5.1
2,matplotlib,3.10.8
3,nbclient,0.10.4
4,nbformat,5.10.4
5,numpy,2.4.0
6,openpyxl,3.1.5
7,pandas,2.3.3
8,scipy,1.17.0


## 5. HiMaLAYAS source and version identification — the critical check

This repo's README pins `himalayas==0.0.15` and ships a vendored reference copy at `himalayas_src/`. The kernel running this notebook may import a *different* HiMaLAYAS install entirely (e.g. an editable install of a sibling development repo). `01_reproduce_submitted_figures.ipynb` must reconcile this before its reproduction result can be trusted.

In [6]:
himalayas_diag = himalayas_diagnostics(layout["repo_root"])
print(json.dumps(himalayas_diag, indent=2))

{
  "imported_version": "0.0.16a0",
  "imported_file": "/Users/irahorecka/Desktop/harddrive_desktop/PhD/University of Toronto/Rost Lab/GitHub/himalayas/src/himalayas/__init__.py",
  "is_editable_install": true,
  "editable_project_location": "/Users/irahorecka/Desktop/harddrive_desktop/PhD/University of Toronto/Rost Lab/GitHub/himalayas",
  "editable_git": {
    "is_git_repo": true,
    "branch": "v0.0.16",
    "commit": "0c511157663c7f392d9338e101830c462ce59b9c",
    "describe": "v0.0.15-4-g0c51115",
    "is_dirty": true,
    "dirty_files": [
      "M src/himalayas/__init__.py",
      "?? PREVIEW_NOTES.md",
      "?? _archive/",
      "?? preview-release-notes.sh",
      "?? test.py"
    ]
  },
  "vendored_copy_path": "/Users/irahorecka/Desktop/harddrive_desktop/PhD/University of Toronto/Rost Lab/GitHub/himalayas-publication/himalayas_src/himalayas/__init__.py",
  "vendored_copy_version": "0.0.15",
  "readme_pinned_version": "0.0.15",
  "matches_readme_pinned_version": false
}


In [7]:
flags = []

if not himalayas_diag["matches_readme_pinned_version"]:
    flags.append(
        "ACTIVE HiMaLAYAS VERSION MISMATCH: kernel imports "
        f"{himalayas_diag['imported_version']!r} but README pins "
        f"{himalayas_diag['readme_pinned_version']!r}."
    )

if himalayas_diag["is_editable_install"]:
    editable_git = himalayas_diag["editable_git"] or {}
    if editable_git.get("is_dirty"):
        flags.append(
            "EDITABLE HiMaLAYAS SOURCE IS DIRTY: "
            f"{himalayas_diag['editable_project_location']} has "
            f"{len(editable_git.get('dirty_files', []))} uncommitted path(s): "
            f"{editable_git['dirty_files']}."
        )
    describe = editable_git.get("describe") or ""
    if "-" in describe:
        flags.append(
            "EDITABLE HiMaLAYAS SOURCE IS AHEAD OF ITS LAST TAG: " f"git describe = {describe!r}."
        )

if (
    himalayas_diag["vendored_copy_version"]
    and himalayas_diag["vendored_copy_version"] != himalayas_diag["imported_version"]
):
    flags.append(
        "VENDORED COPY VERSION DIFFERS FROM IMPORTED VERSION: "
        f"himalayas_src/ declares {himalayas_diag['vendored_copy_version']!r}, "
        f"kernel imports {himalayas_diag['imported_version']!r}."
    )

if pub_git["is_dirty"]:
    flags.append(
        "PUBLICATION REPO HAS UNCOMMITTED/UNTRACKED PATHS: "
        f"{len(pub_git['dirty_files'])} path(s) -- expected during active revision "
        "work (e.g. this new revision/ tree itself); noted for provenance only."
    )

print(f"{len(flags)} flag(s) raised:\n")
for f in flags:
    print(f"- {f}\n")

5 flag(s) raised:

- ACTIVE HiMaLAYAS VERSION MISMATCH: kernel imports '0.0.16a0' but README pins '0.0.15'.

- EDITABLE HiMaLAYAS SOURCE IS DIRTY: /Users/irahorecka/Desktop/harddrive_desktop/PhD/University of Toronto/Rost Lab/GitHub/himalayas has 5 uncommitted path(s): ['M src/himalayas/__init__.py', '?? PREVIEW_NOTES.md', '?? _archive/', '?? preview-release-notes.sh', '?? test.py'].

- EDITABLE HiMaLAYAS SOURCE IS AHEAD OF ITS LAST TAG: git describe = 'v0.0.15-4-g0c51115'.

- VENDORED COPY VERSION DIFFERS FROM IMPORTED VERSION: himalayas_src/ declares '0.0.15', kernel imports '0.0.16a0'.

- PUBLICATION REPO HAS UNCOMMITTED/UNTRACKED PATHS: 6 path(s) -- expected during active revision work (e.g. this new revision/ tree itself); noted for provenance only.



## 6. Input data inventory and file hashes

In [8]:
INPUT_SPECS = [
    {"path": "data/yeast/gi_pcc_sampled.tsv", "referenced_by": ["fig_1.ipynb"]},
    {"path": "data/yeast/gi_score_sampled.tsv", "referenced_by": ["supp_fig_1.ipynb"]},
    {
        "path": "data/yeast/go_bp_name_to_orfs.json",
        "referenced_by": ["fig_1.ipynb", "supp_fig_1.ipynb"],
    },
    {"path": "data/yeast/yeast_essential_orfs.txt", "referenced_by": ["supp_fig_1.ipynb"]},
    {
        "path": "data/interdisciplinary/WorldWideDishes_2024_June.xlsx",
        "referenced_by": ["supp_fig_2.ipynb"],
    },
]

input_rows = []
for spec in INPUT_SPECS:
    full_path = layout["repo_root"] / spec["path"]
    exists = full_path.exists()
    input_rows.append(
        {
            "path": spec["path"],
            "referenced_by": ", ".join(spec["referenced_by"]),
            "exists": exists,
            "size_bytes": full_path.stat().st_size if exists else None,
            "mtime_utc": (
                datetime.fromtimestamp(full_path.stat().st_mtime, tz=timezone.utc).strftime(
                    "%Y-%m-%dT%H:%M:%SZ"
                )
                if exists
                else None
            ),
            "sha256": sha256_file(full_path) if exists else None,
        }
    )

missing_inputs = [r["path"] for r in input_rows if not r["exists"]]
if missing_inputs:
    flags.append(f"MISSING INPUT FILES: {missing_inputs}")

pd.DataFrame(input_rows)

,path,referenced_by,exists,size_bytes,mtime_utc,sha256
0,data/yeast/gi_pcc_sampled.tsv,fig_1.ipynb,True,7047963,2026-01-20T14:09:38Z,29990c13b2926409d474da36388e084f905702ee540b7c...
1,data/yeast/gi_score_sampled.tsv,supp_fig_1.ipynb,True,7396476,2026-01-20T14:10:57Z,af28d767c88d87fd441ff97e31167e717c059c63e82c6e...
2,data/yeast/go_bp_name_to_orfs.json,"fig_1.ipynb, supp_fig_1.ipynb",True,347454,2026-01-23T14:15:12Z,c2f90b8a657cee82879b030743b86cb9e0d4c9491b2100...
3,data/yeast/yeast_essential_orfs.txt,supp_fig_1.ipynb,True,15926,2024-04-30T19:46:32Z,74c1b30d1dc9034a1c995ff6e123d403b4a542739a1d9f...
4,data/interdisciplinary/WorldWideDishes_2024_Ju...,supp_fig_2.ipynb,True,312073,2025-12-25T15:12:29Z,6fcea11be9c4973d6dfaaa8202b9577bce0f52d0d51ff2...


## 7. Random seed policy

Root notebooks are checked directly (not assumed) for any random-seed or stochastic-parameter usage.

In [9]:
ROOT_NOTEBOOKS = ["fig_1.ipynb", "supp_fig_1.ipynb", "supp_fig_2.ipynb"]
SEED_PATTERN = re.compile(r"random_state|np\.random|random\.seed|\bseed\s*=", re.IGNORECASE)

root_notebook_seed_hits = []
for nb_name in ROOT_NOTEBOOKS:
    nb_json = json.loads((layout["repo_root"] / nb_name).read_text())
    for i, cell in enumerate(nb_json["cells"]):
        if cell["cell_type"] != "code":
            continue
        src = "".join(cell["source"])
        if SEED_PATTERN.search(src):
            root_notebook_seed_hits.append({"notebook": nb_name, "cell_index": i})

print(
    f"Random-seed / stochastic-parameter references found in root notebooks: "
    f"{len(root_notebook_seed_hits)}"
)
if root_notebook_seed_hits:
    display(pd.DataFrame(root_notebook_seed_hits))
else:
    print(
        "None found -- consistent with the submitted analyses being deterministic "
        "(Ward/Euclidean linkage with optimal_ordering=True takes no random seed)."
    )

print(f"\nDEFAULT_RANDOM_SEED for stochastic revision notebooks (10/20/21): {DEFAULT_RANDOM_SEED}")

Random-seed / stochastic-parameter references found in root notebooks: 0
None found -- consistent with the submitted analyses being deterministic (Ward/Euclidean linkage with optimal_ordering=True takes no random seed).

DEFAULT_RANDOM_SEED for stochastic revision notebooks (10/20/21): 0


## 8. Assemble and write manifest

In [10]:
manifest = {
    "notebook": "00_environment_check.ipynb",
    "generated_at_utc": __import__("revision_utils").utc_timestamp(),
    "repo_root": str(layout["repo_root"]),
    "publication_repo_git": pub_git,
    "python": python_info_result,
    "dependency_versions": dep_versions,
    "himalayas": himalayas_diag,
    "flags": flags,
    "input_files": input_rows,
    "random_seed_policy": {
        "default_random_seed": DEFAULT_RANDOM_SEED,
        "policy": (
            "Submitted baseline analyses (root fig_1/supp_fig_1/supp_fig_2 notebooks) use "
            "deterministic Ward/Euclidean clustering with optimal_ordering=True and require "
            "no random seed. Revision notebooks introducing randomness (noise perturbation, "
            "permutation nulls, cluster-label randomization) must seed from DEFAULT_RANDOM_SEED "
            "unless a notebook-specific seed is documented in that notebook's own header, and "
            "must record the seed actually used in that notebook's manifest."
        ),
        "observed_seed_usage_in_root_notebooks": root_notebook_seed_hits,
    },
}

manifest_path = layout["manifests_dir"] / "00_environment_check_manifest.json"
write_manifest(manifest, manifest_path)
print(f"Manifest written to: {manifest_path}")

# Round-trip check.
reloaded = json.loads(manifest_path.read_text())
assert reloaded == json.loads(json.dumps(manifest, default=str)), "manifest round-trip mismatch"
print("Manifest JSON round-trip verified.")

Manifest written to: /Users/irahorecka/Desktop/harddrive_desktop/PhD/University of Toronto/Rost Lab/GitHub/himalayas-publication/revision/outputs/manifests/00_environment_check_manifest.json
Manifest JSON round-trip verified.


## 9. Readiness summary

In [11]:
print("=" * 72)
print("READINESS SUMMARY -- 00_environment_check")
print("=" * 72)
print(f"Repo root:              {layout['repo_root']}")
print(
    f"Publication repo:       branch={pub_git.get('branch')} "
    f"commit={(pub_git.get('commit') or '')[:10]} dirty={pub_git.get('is_dirty')}"
)
print(
    f"Python:                 {python_info_result['version'].split()[0]}  ({python_info_result['executable']})"
)
print(
    f"HiMaLAYAS imported:     {himalayas_diag['imported_version']} "
    f"({'editable install' if himalayas_diag['is_editable_install'] else 'installed package'})"
)
print(f"  from:                 {himalayas_diag['imported_file']}")
print(f"README-pinned version:  {himalayas_diag['readme_pinned_version']}")
print(f"Vendored copy version:  {himalayas_diag['vendored_copy_version']}")
print(f"Version match:          {himalayas_diag['matches_readme_pinned_version']}")
print()
print(f"Input files located:    {sum(1 for r in input_rows if r['exists'])}/{len(input_rows)}")
print(f"Manifest written to:    {manifest_path}")
print()

if flags:
    print(f"FLAGS RAISED ({len(flags)}) -- resolve or explicitly acknowledge before notebook 01:")
    for f in flags:
        print(f"  - {f}")
else:
    print("No flags raised.")
print()

print("Recommendation for 01_reproduce_submitted_figures.ipynb:")
if not himalayas_diag["matches_readme_pinned_version"]:
    print(
        "  Do NOT treat a figure reproduction under the current active environment as final.\n"
        "  Either (a) reinstall the pinned `himalayas==0.0.15` release before running notebook 01,\n"
        "  or (b) run under the current development version AND explicitly record the deviation\n"
        "  plus any figure differences against the pinned version in notebook 01's reproduction notes."
    )
else:
    print("  Environment matches the README-pinned version; proceed to notebook 01.")

READINESS SUMMARY -- 00_environment_check
Repo root:              /Users/irahorecka/Desktop/harddrive_desktop/PhD/University of Toronto/Rost Lab/GitHub/himalayas-publication
Publication repo:       branch=first-revision commit=eaeb0b63c5 dirty=True
Python:                 3.12.8  (/Users/irahorecka/.pyenv/versions/3.12.8/envs/himalayas/bin/python)
HiMaLAYAS imported:     0.0.16a0 (editable install)
  from:                 /Users/irahorecka/Desktop/harddrive_desktop/PhD/University of Toronto/Rost Lab/GitHub/himalayas/src/himalayas/__init__.py
README-pinned version:  0.0.15
Vendored copy version:  0.0.15
Version match:          False

Input files located:    5/5
Manifest written to:    /Users/irahorecka/Desktop/harddrive_desktop/PhD/University of Toronto/Rost Lab/GitHub/himalayas-publication/revision/outputs/manifests/00_environment_check_manifest.json

FLAGS RAISED (5) -- resolve or explicitly acknowledge before notebook 01:
  - ACTIVE HiMaLAYAS VERSION MISMATCH: kernel imports '0.0.16a